# Launch Sites Locations Analysis with Folium

## Overview
The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site (e.g., the initial position of rocket trajectories, proximity to coastlines, railways, highways, and populated cities).

### Objectives
- **Task 1:** Mark all launch sites on a Folium map using `folium.Circle` and `folium.Marker`.
- **Task 2:** Mark the success/failed launches for each site on the map using `MarkerCluster` and color-coded markers.
- **Task 3:** Calculate the distances between a launch site and its proximities (coastline, railway, highway, and city) and visualize them with `folium.PolyLine`.

In [1]:
import folium
import pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians

# Load the SpaceX launch geo dataset
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
spacex_df = pd.read_csv(URL)
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()[['Launch Site', 'Lat', 'Long']]
launch_sites_df

 Launch Site       Lat        Long
 CCAFS LC-40 28.562302  -80.577356
CCAFS SLC-40 28.563197  -80.576820
  KSC LC-39A 28.573255  -80.646895
 VAFB SLC-4E 34.632834 -120.610745


### Task 1: Mark all launch sites on a map
We initialize a `folium.Map` centered at NASA Johnson Space Center (`[29.559684888503615, -95.0830971930759]`) and add a 1,000-meter `folium.Circle` and a text-labeled `folium.Marker` for each launch site.

In [2]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# Add a Circle and Marker for each launch site
for index, row in launch_sites_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    site_name = row['Launch Site']
    
    circle = folium.Circle(
        coordinate,
        radius=1000,
        color='#d35400',
        fill=True
    ).add_child(folium.Popup(site_name))
    
    marker = folium.map.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html=f'<div style="font-size: 12; color:#d35400;"><b>{site_name}</b></div>'
        )
    )
    site_map.add_child(circle)
    site_map.add_child(marker)

print('Added Circle and Marker objects for all launch sites.')
site_map

Added Circle and Marker objects for all launch sites.


### Task 2: Mark the success/failed launches for each site on the map
Next, we create a `MarkerCluster` and assign a `marker_color` (`green` for `class == 1` success, `red` for `class == 0` failure) to visualize launch outcomes at each complex.

In [3]:
# Assign marker colors based on launch outcome class
spacex_df['marker_color'] = spacex_df['class'].apply(lambda x: 'green' if x == 1 else 'red')

marker_cluster = MarkerCluster()
site_map.add_child(marker_cluster)

for index, record in spacex_df.iterrows():
    marker = folium.Marker(
        location=[record['Lat'], record['Long']],
        icon=folium.Icon(color='white', icon_color=record['marker_color']),
        popup=f"{record['Launch Site']} - {'Success' if record['class'] == 1 else 'Failure'}"
    )
    marker_cluster.add_child(marker)

print(f'Added {len(spacex_df)} color-coded launch outcome markers to MarkerCluster.')
site_map

Added 56 color-coded launch outcome markers to MarkerCluster.


### Task 3: Calculate the distances between a launch site to its proximities
We add a `MousePosition` plugin to inspect coordinates on the map, compute distances using the Haversine formula, and draw `folium.PolyLine` segments from `CCAFS LC-40` to the nearest coastline, railway, highway, and city (Cocoa Beach).

In [4]:
# Add Mouse Position plugin to inspect coordinates interactively
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)
site_map.add_child(mouse_position)

# Haversine distance function in kilometers
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

# Coordinates for CCAFS LC-40 and nearby features
launch_site_coord = [28.562302, -80.577356]
proximities = {
    'Coastline': [28.56367, -80.56798],
    'Railway':   [28.57206, -80.58525],
    'Highway':   [28.56335, -80.57085],
    'City':      [28.39220, -80.60770]
}

for label, coord in proximities.items():
    distance = calculate_distance(launch_site_coord[0], launch_site_coord[1], coord[0], coord[1])
    print(f'Distance to {label}: {distance:.2f} km')
    
    # Add distance label marker
    distance_marker = folium.Marker(
        coord,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html=f'<div style="font-size: 12; color:#d35400;"><b>{label}: {distance:.2f} KM</b></div>'
        )
    )
    # Draw polyline connecting launch site to proximity feature
    line = folium.PolyLine(locations=[launch_site_coord, coord], weight=2, color='blue')
    site_map.add_child(distance_marker)
    site_map.add_child(line)

site_map

Distance to Coastline: 0.93 km
Distance to Railway:   1.33 km
Distance to Highway:   0.65 km
Distance to City:      19.15 km


### Key Findings
- **Proximity to Coastlines:** All launch sites sit within ~0.9 km of the coastline so rocket trajectories head directly over open ocean, minimizing risk to populated areas.
- **Proximity to Transport Infrastructure:** Launch pads are located close to railways (~1.3 km) and highways (~0.6 km) to support heavy booster and payload logistics.
- **Distance from Cities:** Launch complexes are situated ~19 km away from major population centers like Cocoa Beach for public safety.